# SAMI — Notebook 3 · Clustering de usuarios, clasificación y sentimiento

**NLP simplificado a nivel usuario.** KMeans sobre embeddings de oraciones (primario) y TF-IDF de texto lematizado (comparación); contraste contra la **clasificación original** de la base (Chat_summary → 7 categorías MMC); sentimiento como señal de malestar; y el **mapa síntesis** de necesidad + tono por ciudad como cierre de todo el análisis.

*Se abandonaron los métodos más pesados del pipeline anterior (UMAP/HDBSCAN, zero-shot tinting, temas emergentes, reformulación por embeddings, emoción de 7 clases).*

## 0. Setup, data & original classification

In [ ]:
# Imports. Collapsed on purpose -- no analysis here.
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = 0 if DEVICE == "cuda" else -1

In [ ]:
# Temporary neutral styling -- brand palette removed; matplotlib defaults for now.
# These names shadow the old palette so moved cells run unchanged. Re-add the
# real palette later by replacing THIS cell (nothing else references the brand).
_CYCLE = plt.rcParams["axes.prop_cycle"].by_key()["color"]   # matplotlib defaults
PRIMARY = _CYCLE[0]
BLUE_SEQ = BLUES = _CYCLE
EARTH    = _CYCLE
CAT      = _CYCLE
# brand color names -> matplotlib defaults, so cells referencing them still run
AGUA, ARBOL, AMEBA, MADERA, HONGO, NEGRO = (
    _CYCLE[0], _CYCLE[2], _CYCLE[4], _CYCLE[1], "#dddddd", "black")
INK = INK2 = "black"
MUTED = "gray"
GRID  = "#cccccc"
SURFACE = "white"

def cat_colors(n):
    """n distinct matplotlib default colors (categorical)."""
    return [_CYCLE[i % len(_CYCLE)] for i in range(n)]

def bar_colors(n):
    """alias of cat_colors -- n distinct matplotlib default colors."""
    return [_CYCLE[i % len(_CYCLE)] for i in range(n)]

def seq_colors(n):
    """single default color repeated (ordered magnitude -- one hue for now)."""
    return [_CYCLE[0]] * n

def pct_count_autopct(values, min_pct=3.0):
    """Pie label 'xx.x%\n(n)'; blank under min_pct. Label formatter, not color."""
    total = float(sum(values))
    def _fmt(pct):
        if pct < min_pct:
            return ""
        return f"{pct:.1f}%\n({int(round(pct/100*total))})"
    return _fmt

In [ ]:
# Data loaders -- inlined so the notebook is fully self-contained (no external module).
import re, unicodedata

DATA_DIR = "../data_&_docs"
RESPONSES_PATH = DATA_DIR + "/MMC_bot_responses_1783087815.xlsx"
MEAL_PATH = DATA_DIR + "/MMC_MEAL_1783087939.xlsx"
DATA_HEADER_ROW = 2  # header is the 3rd row of the export

def _phone(name):
    return re.sub(r"\D", "", str(name))

# 10 MMC priority cities + common variants -> canonical display name
_CITY_CANON = {
    "medellin": "Medellín", "medellin antioquia": "Medellín", "belen": "Medellín",
    "bogota": "Bogotá", "bogota dc": "Bogotá",
    "cucuta": "Cúcuta",
    "barranquilla": "Barranquilla",
    "santa marta": "Santa Marta",
    "cali": "Cali",
    "cartagena": "Cartagena",
    "bucaramanga": "Bucaramanga",
    "ipiales": "Ipiales",
    "riohacha": "Riohacha", "maicao": "Maicao",
    "soacha": "Soacha", "soacha cundinamarca": "Soacha",
    "necocli": "Necoclí",
}
_NON_CITY = {"colombia", "cundinamarca", "antioquia", "otra", "nan"}

def _fold(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.strip().lower()

def city_canon(name):
    if name is None:
        return "Otra"
    key = _fold(name)
    if key in _NON_CITY or key == "":
        return "Otra"
    if key in _CITY_CANON:
        return _CITY_CANON[key]
    for k, v in _CITY_CANON.items():          # "<city> <extra>" tails
        if key.startswith(k):
            return v
    return "Otra"

def clean_city(raw_city, city_other):
    raw = ("" if raw_city is None else str(raw_city)).strip()
    other = "" if city_other is None or pd.isna(city_other) else str(city_other).strip()
    if raw == "Otra" and other:
        return other.title()
    return raw

def _read_whatsapp(path):
    d = pd.read_excel(path, header=DATA_HEADER_ROW)
    d = d[d["Name"].astype(str).str.startswith("whatsapp")].copy()
    d.reset_index(drop=True, inplace=True)
    return d

def load_responses(path=RESPONSES_PATH):
    d = _read_whatsapp(path)
    d["phone"] = d["Name"].map(_phone)
    d["city_clean"] = [clean_city(c, o) for c, o in zip(d["City"], d["City_other"])]
    d["city_canon"] = d["city_clean"].map(city_canon)
    d["age_num"] = pd.to_numeric(d["Age"], errors="coerce")
    d["ts"] = pd.to_datetime(d["Timestamp"], errors="coerce", utc=True).dt.tz_localize(None)
    d["n_questions"] = pd.to_numeric(d["Questions per user"], errors="coerce")
    return d

def load_meal(path=MEAL_PATH):
    d = _read_whatsapp(path)
    d["phone"] = d["Name"].map(_phone)
    cols = list(d.columns)
    d = d.rename(columns={cols[2]: "utility", cols[3]: "would_recommend",
                          cols[4]: "recommendation", cols[5]: "heard_channel",
                          cols[6]: "heard_medium"})
    d["ts"] = pd.to_datetime(d["Timestamp"], errors="coerce", utc=True).dt.tz_localize(None)
    return d

_NOISE = {"undefined", "?", ""}
def _is_noise(t):
    t = t.strip()
    return len(t) < 3 or t.isdigit() or t.lower() in _NOISE

def load_messages(d=None):
    """Explode the per-user `Messages` blob into one row per user turn."""
    if d is None:
        d = load_responses()
    carry = ["phone", "city_clean", "city_canon", "ts", "Gender",
             "Age Ranges", "Nationality", "age_num"]
    carry = [c for c in carry if c in d.columns]
    rows = []
    for _, r in d.iterrows():
        blob = r.get("Messages")
        if not isinstance(blob, str):
            continue
        parts = [p.strip() for p in blob.split("\n")]
        parts = [p for p in parts if not _is_noise(p)]
        for i, p in enumerate(parts):
            row = {c: r[c] for c in carry}
            row["msg_idx"] = i
            row["n_msgs_user"] = len(parts)
            row["message"] = p
            rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
# The 7 official MMC categories, each with a short Spanish hypothesis. The
# category KEYS (English) are what gets displayed on every chart; the VALUES
# stay in Spanish because they are aligned with the bot's original Chat_summary labels. Short labels were chosen by measuring agreement against the bot's
# own Chat_summary labels (see the validation cell in Section 3): they gave the
# best balanced accuracy — long keyword-stuffed descriptions collapsed onto one or two categories.
MMC_LABELS = {
    "legal documentation":     "documentación legal y trámites migratorios",
    "humanitarian assistance": "ayuda humanitaria",
    "employment":              "empleo",
    "services":                "salud y servicios",
    "protection":              "protección y seguridad",
    "journey information":     "información de viaje",
    "organization search":     "búsqueda de una organización",
}
MMC_CATS = list(MMC_LABELS)
COURTESY = "courtesy / non-substantive"
NOISE    = "unclustered"
FRAG     = "short fragment"                 # location/greeting fragments, not a theme
NON_TOPIC_CATS = [COURTESY, NOISE, FRAG]    # excluded from topic cross-cuts

# one distinct colour per category (brand colours first, then muted extras)
CAT_COLORS = {
    "legal documentation":     AGUA,
    "humanitarian assistance": ARBOL,
    "employment":              AMEBA,
    "services":                MADERA,
    "protection":              "#c9788f",
    "journey information":     "#4a9d9c",
    "organization search":     "#d9a45b",
    "emergent":                "#8e5572",
    COURTESY:                  HONGO,
    NOISE:                     "#cfcfcf",
    FRAG:                      "#b0b0b0",
}
def cat_palette(cats):
    return [CAT_COLORS.get(c, NEGRO) for c in cats]

In [ ]:
df   = load_responses()          # one row per user (has Chat_summary)
meal = load_meal()               # MEAL survey (satisfaction)
msgs = load_messages(df)         # one row per user message (the spine)
print(f"users: {len(df)}  |  messages: {len(msgs)}  |  device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

### 0.1 Original classification (from the bot's `Chat_summary`)

The bot already tags each user with a `Chat_summary` of MMC categories — the database's **original classification**. We map it to the 7 canonical categories and attach one dominant category per user (broadcast to their messages). This is the ground truth the KMeans clusters are checked against later. *Per-message nuance is lost — every message of a user shares that user's dominant category.*

In [ ]:
TOPIC_MAP_LC = {
    "legal documentation":     "legal documentation",
    "humanitarian assistance": "humanitarian assistance",
    "employment":              "employment",
    "services":                "services",
    "protection":              "protection",
    "organization search":     "organization search",
    "journey information":     "journey information",
}
UNCLASSIFIED = "unclassified"

def _user_category(summary):
    """Dominant canonical MMC category from a user's Chat_summary free text."""
    if not isinstance(summary, str) or "Use exactly one of these hashtags" in summary:
        return UNCLASSIFIED
    cats = []
    for t in summary.replace("''", ",").split(","):
        t = t.strip().lstrip("#").replace("_", " ").lower()
        if t in TOPIC_MAP_LC:
            cats.append(TOPIC_MAP_LC[t])
    if not cats:
        return UNCLASSIFIED
    return pd.Series(cats).value_counts().index[0]

df["mmc_category"] = df["Chat_summary"].map(_user_category)
_cat_by_phone = df.set_index("phone")["mmc_category"]
msgs["mmc_category"] = msgs["phone"].map(_cat_by_phone).fillna(UNCLASSIFIED)

CAT_COLORS[UNCLASSIFIED] = "#cfcfcf"
NON_TOPIC_CATS = NON_TOPIC_CATS + [UNCLASSIFIED]
print(msgs["mmc_category"].value_counts())

## 1. What users need — by the original classification

*Descriptive slices of the DB's own categories. The interpretable, named view of demand, before we ask whether unsupervised clustering recovers it.*

### 1.1 Category mix by city

In [ ]:
# top cities by message volume (needed by the category-mix chart below)
top_cities = msgs.loc[msgs["city_canon"] != "Otra", "city_canon"].value_counts().head(10)

In [ ]:
cats = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]["mmc_category"]
        .value_counts().head(6).index)
sub = msgs[msgs["city_canon"].isin(top_cities.index) & msgs["mmc_category"].isin(cats)]
mix = (pd.crosstab(sub["city_canon"], sub["mmc_category"], normalize="index")
       .reindex(index=top_cities.index, columns=cats, fill_value=0))
mix.plot(kind="barh", stacked=True, figsize=(10, 6), color=cat_palette(mix.columns))
plt.title("Category mix by city (share of messages)")
plt.xlabel("share"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()

### 1.2 Categories over time

In [ ]:
EVENTS = {
    # "2026-04-15": "example policy milestone",
}
weekly = msgs.dropna(subset=["ts"]).set_index("ts").resample("W").size()
fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(weekly.index, weekly.values, color=BLUE_SEQ[0], alpha=0.5)
ax.plot(weekly.index, weekly.values, color=PRIMARY, linewidth=1.8, marker="o", ms=3)
ax.grid(True, axis="y")
for date, label in EVENTS.items():
    ax.axvline(pd.Timestamp(date), color=MADERA, ls="--", lw=1)
    ax.text(pd.Timestamp(date), weekly.max(), label, rotation=90, va="top", fontsize=8)
ax.set_title("Weekly message volume"); ax.set_ylabel("messages")
plt.tight_layout()

In [ ]:
cats4 = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]["mmc_category"]
         .value_counts().head(4).index)
trend = (msgs.dropna(subset=["ts"])
         .assign(week=lambda d: d["ts"].dt.to_period("W").dt.start_time)
         [lambda d: d["mmc_category"].isin(cats4)]
         .groupby(["week", "mmc_category"]).size().unstack(fill_value=0)
         .reindex(columns=cats4, fill_value=0))
fig, ax = plt.subplots(figsize=(11, 5))
for c in trend.columns:
    ax.plot(trend.index, trend[c], marker="o", ms=3, color=CAT_COLORS[c], label=c)
ax.set_title("Top-4 categories over time (weekly)"); ax.legend(fontsize=8)
plt.tight_layout()

### 1.3 Category by demographics

In [ ]:
top_cats = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]["mmc_category"]
            .value_counts().head(6).index)

def cat_heatmap(group_col, top_n=6, title=""):
    groups = msgs[group_col].value_counts().head(top_n).index
    sub = msgs[msgs[group_col].isin(groups) & msgs["mmc_category"].isin(top_cats)]
    mat = (pd.crosstab(sub[group_col], sub["mmc_category"], normalize="index")
           .reindex(index=groups, columns=top_cats, fill_value=0))
    # Height scales with the number of rows; width is fixed wide enough for the
    # long category names once they are angled.
    fig, ax = plt.subplots(figsize=(11, 0.7 * len(groups) + 2.8))
    sns.heatmap(mat, cmap=sns.light_palette(BLUES[5], as_cmap=True),
                annot=True, fmt=".0%", ax=ax, cbar_kws={"label": "share"})
    ax.set_title(title, pad=12)
    ax.set_xlabel(""); ax.set_ylabel("")
    # Angle the long category names so they stop overlapping; keep the group
    # labels (nationality / age range) horizontal and readable.
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    plt.tight_layout()

cat_heatmap("Nationality", title="Category mix by nationality")

In [ ]:
cat_heatmap("Gender", top_n=4, title="Category mix by gender")

In [ ]:
cat_heatmap("Age Ranges", top_n=4, title="Category mix by age range")

### 1.4 Where users drop off, by category

In [ ]:
last = msgs[msgs["msg_idx"] == msgs["n_msgs_user"] - 1]
last_cat = last.loc[~last["mmc_category"].isin([NOISE]), "mmc_category"].value_counts().head(8)
fig, ax = plt.subplots(figsize=(8, 5))
o = last_cat.iloc[::-1]
ax.barh(o.index, o.values, color=cat_palette(o.index), edgecolor=NEGRO, lw=.5)
ax.set_title("Category of each user's LAST message (where they stopped)")
ax.set_xlabel("users"); plt.tight_layout()
print(f"users ending on a courtesy message: "
      f"{(last['mmc_category']==COURTESY).mean()*100:.1f}%")

### 1.5 Satisfaction (MEAL) × category

**What this shows.** MEAL utility ratings joined to the category each user mostly asked about. **Why it matters.** It flags which needs produce the worst experience.

> **Technical note — small overlap.** Only users who completed the MEAL survey appear (dozens). Each user is assigned their dominant message category, joined on WhatsApp phone number. Read proportions as directional.

In [ ]:
UTIL_ORDER = ["Nada útil", "Medianamente útil", "Útil", "Muy útil"]
dom_cat = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]
           .groupby("phone")["mmc_category"].agg(lambda s: s.value_counts().index[0])
           .rename("dom_cat").reset_index())
merged = dom_cat.merge(meal[["phone", "utility"]], on="phone", how="inner")
merged = merged[merged["utility"].isin(UTIL_ORDER)]
print(f"users with both a category and a MEAL rating: {len(merged)}")

cats6 = merged["dom_cat"].value_counts().head(6).index
sub = merged[merged["dom_cat"].isin(cats6)]
tabU = (pd.crosstab(sub["dom_cat"], sub["utility"], normalize="index")
        .reindex(index=cats6, columns=UTIL_ORDER, fill_value=0))
tabU.plot(kind="barh", stacked=True, figsize=(10, 6), color=bar_colors(4))
plt.title("Utility rating by dominant category (MEAL respondents)")
plt.xlabel("share"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()

## 2. User-level clustering

*One document per user (all their messages concatenated). Cluster **users**, not messages. Primary features: sentence embeddings. Comparison: TF-IDF of lemmatized text.*

### 2.1 User documents & lemmatization

In [ ]:
# One document per user = all their messages concatenated. Lemmatize (spaCy es)
# for the TF-IDF comparison; the embeddings use the raw text.
import spacy
try:
    nlp = spacy.load("es_core_news_sm", disable=["ner", "parser"])
except OSError:
    from spacy.cli import download as _sp_dl
    _sp_dl("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm", disable=["ner", "parser"])

user_docs = (msgs.groupby("phone")["message"]
             .apply(lambda s: " ".join(s.astype(str))).rename("doc").reset_index())
user_docs["category"] = user_docs["phone"].map(_cat_by_phone).fillna(UNCLASSIFIED)

def _lemmatize(text):
    return " ".join(tok.lemma_.lower() for tok in nlp(text)
                    if tok.is_alpha and not tok.is_stop and len(tok) > 2)

user_docs["lemmas"] = [_lemmatize(t) for t in user_docs["doc"]]
print(f"{len(user_docs)} user documents")
user_docs.head(3)

### 2.2 KMeans on sentence embeddings (primary)

In [ ]:
# Primary features: multilingual-e5-large embeddings of each user document.
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

_embedder = SentenceTransformer("intfloat/multilingual-e5-large", device=DEVICE)
if DEVICE == "cuda":
    _embedder = _embedder.half()
emb = _embedder.encode(["query: " + d for d in user_docs["doc"]],
                       batch_size=16, normalize_embeddings=True, show_progress_bar=False)
emb = np.asarray(emb, dtype="float32")
del _embedder
if DEVICE == "cuda":
    torch.cuda.empty_cache()

K = 7   # match the 7 official MMC categories
km_emb = KMeans(n_clusters=K, random_state=42, n_init=10).fit(emb)
user_docs["cluster_emb"] = km_emb.labels_
print("embeddings:", emb.shape)
print(user_docs["cluster_emb"].value_counts().sort_index().to_string())

### 2.3 Comparison — KMeans on TF-IDF of lemmatized text

In [ ]:
# A lighter, fully interpretable comparison: TF-IDF over lemmatized user docs.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=2000, min_df=2)
X_tfidf = tfidf.fit_transform(user_docs["lemmas"])
km_tfidf = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_tfidf)
user_docs["cluster_tfidf"] = km_tfidf.labels_
print("tfidf matrix:", X_tfidf.shape)
print(user_docs["cluster_tfidf"].value_counts().sort_index().to_string())

## 3. Do the clusters match the original classification?

*How well does unsupervised clustering recover the 7 categories the bot already assigned? ARI / NMI / purity, plus a confusion heatmap.*

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def _purity(clusters, labels):
    ct = pd.crosstab(clusters, labels)
    return ct.max(axis=1).sum() / ct.values.sum()

known = user_docs["category"] != UNCLASSIFIED
truth = user_docs.loc[known, "category"]
rows = []
for name, col in [("embeddings", "cluster_emb"), ("TF-IDF", "cluster_tfidf")]:
    cl = user_docs.loc[known, col]
    rows.append({"features": name,
                 "ARI": adjusted_rand_score(truth, cl),
                 "NMI": normalized_mutual_info_score(truth, cl),
                 "purity": _purity(cl, truth)})
agreement = pd.DataFrame(rows).set_index("features").round(3)
print(f"users with a known original category: {int(known.sum())} / {len(user_docs)}")
agreement

In [ ]:
# Which original category dominates each embedding cluster? (row-normalized)
conf = pd.crosstab(user_docs.loc[known, "cluster_emb"],
                   user_docs.loc[known, "category"], normalize="index")
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(conf, annot=True, fmt=".0%", cmap="Blues", ax=ax, cbar_kws={"label": "share"})
ax.set_title("Embedding clusters vs original category (row-normalized)")
ax.set_xlabel("original category"); ax.set_ylabel("KMeans cluster")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()

## 4. Sentiment — an unsolicited distress signal

*3-class sentiment on every message (not just the few MEAL respondents). The 7-class emotion model from the old pipeline is dropped.*

In [ ]:
# 3-class sentiment per message.
from transformers import pipeline

texts = msgs["message"].astype(str).tolist()
sent_clf = pipeline("sentiment-analysis",
                    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                    device=GPU, truncation=True, max_length=256)

def _norm_sent(label):
    l = label.lower()
    if "neg" in l or l.endswith("0"):
        return "negative"
    if "pos" in l or l.endswith("2"):
        return "positive"
    return "neutral"

preds = sent_clf([t[:256] for t in texts], batch_size=32)
msgs["sentiment"] = [_norm_sent(p["label"]) for p in preds]
del sent_clf
if DEVICE == "cuda":
    torch.cuda.empty_cache()

SENT_ORDER = ["negative", "neutral", "positive"]
SENT_COLORS = {"negative": "#c9788f", "neutral": HONGO, "positive": AGUA}
fig, ax = plt.subplots(figsize=(7, 3.2))
sd = msgs["sentiment"].value_counts().reindex(SENT_ORDER)
ax.barh(SENT_ORDER, sd.values, color=[SENT_COLORS[s] for s in SENT_ORDER],
        edgecolor=NEGRO, lw=.5)
ax.set_title("Message sentiment"); ax.set_xlabel("messages")
plt.tight_layout()
print("sentiment:", dict(sd), f"|  negative share: {sd['negative']/sd.sum()*100:.1f}%")

### 4.1 Sentiment by category

In [ ]:
# Which needs arrive with the most negative tone?
cats6 = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]["mmc_category"]
         .value_counts().head(6).index)
sub = msgs[msgs["mmc_category"].isin(cats6)]
tabS = (pd.crosstab(sub["mmc_category"], sub["sentiment"], normalize="index")
        .reindex(index=cats6, columns=SENT_ORDER, fill_value=0))
tabS.plot(kind="barh", stacked=True, figsize=(10, 5),
          color=[SENT_COLORS[s] for s in SENT_ORDER])
plt.title("Sentiment mix by category (share of messages)")
plt.xlabel("share"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()

## 5. Geographic synthesis — need + tone by city

**The closer of the whole analysis (EDA + General + NLP).** Colombia with, per city, its **dominant need** (MMC category) and the **average tone** of its messages — where a need concentrates and where people write in with the most negative tone, to target field response.

> **Technical note.** One bubble per city (size = messages), positioned from a curated coordinate table; only cities with ≥ 15 mapped messages are shown. Sentiment is the mean per-message polarity (−1 / 0 / +1); category is the most common per city. A light CartoDB Positron basemap grounds the bubbles.

In [ ]:
# City coordinates + per-city aggregates. The map itself uses a contextily
# basemap (no floating shapefile) -- see _basemap() below.
import unicodedata
import contextily as cx

CITY_COORDS = {
    "bogota": (-74.0721, 4.7110),   "medellin": (-75.5812, 6.2442),
    "cali": (-76.5320, 3.4516),     "barranquilla": (-74.7813, 10.9685),
    "cartagena": (-75.4794, 10.3910),"cucuta": (-72.5078, 7.8939),
    "bucaramanga": (-73.1227, 7.1193),"maicao": (-72.2382, 11.3776),
    "riohacha": (-72.9072, 11.5449),"ipiales": (-77.6417, 0.8256),
    "santa marta": (-74.1990, 11.2408),"soacha": (-74.2168, 4.5794),
    "necocli": (-76.7889, 8.4245),
}
def _strip(s):
    s = str(s).strip().lower()
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

msgs["city_key"] = msgs["city_canon"].map(_strip)
matched = msgs["city_key"].isin(CITY_COORDS)
print(f"{msgs.loc[matched,'city_key'].nunique()} cities mapped, "
      f"covering {matched.mean()*100:.0f}% of messages")

MIN_MSGS = 15
SENT_SIGN = {"negative": -1, "neutral": 0, "positive": 1}
gm = msgs[matched & ~msgs["mmc_category"].isin(NON_TOPIC_CATS)].copy()
gm["sent_sign"] = gm["sentiment"].map(SENT_SIGN)
geo = gm.groupby("city_key").agg(msgs_n=("city_key", "size"), sentiment=("sent_sign", "mean"))
geo = geo[geo["msgs_n"] >= MIN_MSGS]
geo["dominant"] = gm.groupby("city_key")["mmc_category"].agg(lambda s: s.value_counts().index[0])
geo["lon"] = [CITY_COORDS[c][0] for c in geo.index]
geo["lat"] = [CITY_COORDS[c][1] for c in geo.index]
geo["label"] = [c.title() for c in geo.index]
geo = geo.sort_values("msgs_n", ascending=False)

def _basemap(ax, title):
    "Frame Colombia and drop a light CartoDB Positron basemap behind the bubbles."
    ax.set_xlim(-80, -66); ax.set_ylim(-4.5, 13)
    ax.set_aspect("equal")
    cx.add_basemap(ax, crs="EPSG:4326", zorder=0,
                   source=cx.providers.CartoDB.PositronNoLabels, attribution_size=6)
    ax.set_title(title); ax.set_axis_off()
def _sizes(frame):
    return 60 + (frame["msgs_n"] / frame["msgs_n"].max()) * 900
print(f"{len(geo)} cities with >= {MIN_MSGS} messages mapped")
display(geo[["label", "msgs_n", "sentiment", "dominant"]].reset_index(drop=True))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
sent_cmap = LinearSegmentedColormap.from_list("mmc_sent", ["#c9788f", HONGO, AGUA])
fig, ax = plt.subplots(figsize=(7, 8))
_basemap(ax, "Mean message sentiment by city")
sc = ax.scatter(geo["lon"], geo["lat"], s=_sizes(geo), c=geo["sentiment"],
                cmap=sent_cmap, vmin=-0.6, vmax=0.6, edgecolor=NEGRO, lw=.6, zorder=2)
for _, r in geo.iterrows():
    ax.annotate(r["label"], (r["lon"], r["lat"]), fontsize=7, xytext=(4, 4),
                textcoords="offset points")
cbar = plt.colorbar(sc, ax=ax, shrink=0.5); cbar.set_label("mean polarity (−1 … +1)")
ax.text(0.01, 0.01, "bubble size = messages", transform=ax.transAxes, fontsize=8)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
_basemap(ax, "Dominant category by city")
sizes = _sizes(geo)
for cat in geo["dominant"].unique():
    m = geo["dominant"] == cat
    ax.scatter(geo.loc[m, "lon"], geo.loc[m, "lat"], s=sizes[m],
               color=CAT_COLORS.get(cat, NEGRO), edgecolor=NEGRO, lw=.6, label=cat, zorder=2)
for _, r in geo.iterrows():
    ax.annotate(r["label"], (r["lon"], r["lat"]), fontsize=7, xytext=(4, 4),
                textcoords="offset points")
ax.legend(loc="lower left", fontsize=8, framealpha=.9, title="dominant category")
ax.text(0.01, 0.01, "bubble size = messages", transform=ax.transAxes, fontsize=8)
plt.tight_layout()